# 02 -- Sockets TCP/UDP

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre le modele client-serveur et les couches reseau ;
- creer un serveur et un client TCP avec le module `socket` ;
- creer un serveur et un client UDP ;
- gerer les connexions multiples avec `selectors` ;
- utiliser `socketserver` pour simplifier ;
- comprendre l'encodage des donnees (bytes, struct) ;
- gerer les erreurs reseau et les timeouts.

## Prerequis -- ce que vous devez maitriser

- les fonctions Python, les context managers (`with`) ;
- les notions de base du reseau (IP, port, TCP vs UDP) ;
- `subprocess` pour lancer des processus (voir `01_Subprocess_et_shell.ipynb`) ;

Ce que ce notebook **ne couvrira pas** :

- HTTP (voir `03_HTTP_avec_httpx.ipynb`) ;
- les frameworks web (voir `Web_backend/`).

## Plan

1. Rappels reseau
2. Module `socket` -- bases
3. Serveur TCP echo
4. Client TCP
5. Protocol avec delimiteur
6. Serveur UDP
7. Client UDP
8. `selectors` -- multiplexage I/O
9. `socketserver` -- serveur simplifie
10. `struct` -- encodage binaire
11. Synthese
12. Exercices

---

## 1. Rappels reseau

| Couche | Protocole | Proprietes |
|---|---|---|
| Transport | TCP | Fiable, ordonne, connexion |
| Transport | UDP | Non fiable, sans connexion, rapide |
| Application | HTTP, DNS, SMTP | Protocoles de haut niveau |

**TCP** garantit la livraison et l'ordre. **UDP** est plus rapide mais sans garantie.

In [ ]:
import socket

---

## 2. Module `socket` -- bases

In [ ]:
# Creer un socket TCP
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
print(type(sock))
sock.close()

In [ ]:
# Creer un socket UDP
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.close()

`AF_INET` = IPv4, `SOCK_STREAM` = TCP, `SOCK_DGRAM` = UDP.

### Obtenir des informations reseau

In [ ]:
socket.gethostname()

In [ ]:
socket.gethostbyname('localhost')

---

## 3. Serveur TCP echo

Un serveur echo renvoie tout ce qu'il recoit. Voici le pattern classique :

In [ ]:
import socket
import threading

def tcp_echo_server(host: str = '127.0.0.1', port: int = 0) -> tuple[str, int]:
    """Lance un serveur TCP echo dans un thread. Retourne (host, port)."""
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port))
    srv.listen(5)
    actual_port = srv.getsockname()[1]

    def handle():
        conn, addr = srv.accept()
        with conn:
            data = conn.recv(1024)
            if data:
                conn.sendall(data)
        srv.close()

    threading.Thread(target=handle, daemon=True).start()
    return host, actual_port

host, port = tcp_echo_server()
print(f'Serveur ecoute sur {host}:{port}')

---

## 4. Client TCP

In [ ]:
def tcp_echo_client(host: str, port: int, message: str) -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((host, port))
        s.sendall(message.encode('utf-8'))
        data = s.recv(1024)
    return data.decode('utf-8')

reponse = tcp_echo_client(host, port, 'Bonjour TCP!')
reponse

---

## 5. Protocole avec delimiteur

TCP est un flux d'octets, pas de messages. Il faut definir un protocole pour delimiter les messages.

In [ ]:
def send_message(sock: socket.socket, msg: str) -> None:
    """Envoie un message precede de sa longueur sur 4 octets."""
    data = msg.encode('utf-8')
    length = len(data).to_bytes(4, 'big')
    sock.sendall(length + data)

def recv_message(sock: socket.socket) -> str:
    """Recoit un message precede de sa longueur."""
    raw_length = sock.recv(4)
    if not raw_length:
        return ''
    length = int.from_bytes(raw_length, 'big')
    data = b''
    while len(data) < length:
        chunk = sock.recv(length - len(data))
        if not chunk:
            raise ConnectionError('Connexion fermee')
        data += chunk
    return data.decode('utf-8')

Ce pattern (prefixe de longueur) est utilise par de nombreux protocoles binaires.

---

## 6. Serveur UDP

In [ ]:
def udp_echo_server(host: str = '127.0.0.1', port: int = 0) -> tuple[str, int]:
    srv = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    srv.bind((host, port))
    actual_port = srv.getsockname()[1]

    def handle():
        data, addr = srv.recvfrom(1024)
        srv.sendto(data, addr)
        srv.close()

    threading.Thread(target=handle, daemon=True).start()
    return host, actual_port

udp_host, udp_port = udp_echo_server()
print(f'Serveur UDP sur {udp_host}:{udp_port}')

---

## 7. Client UDP

In [ ]:
def udp_echo_client(host: str, port: int, message: str) -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.sendto(message.encode('utf-8'), (host, port))
        data, _ = s.recvfrom(1024)
    return data.decode('utf-8')

udp_echo_client(udp_host, udp_port, 'Bonjour UDP!')

---

## 8. `selectors` -- multiplexage I/O

Pour gerer plusieurs connexions simultanees sans threads, on utilise `selectors`.

In [ ]:
import selectors

sel = selectors.DefaultSelector()
print(type(sel))

Le pattern `selectors` permet de surveiller plusieurs sockets et de reagir quand l'un est pret en lecture ou ecriture. C'est la base de l'event loop d'asyncio.

---

## 9. `socketserver` -- serveur simplifie

`socketserver` fournit des classes de base pour ecrire des serveurs TCP/UDP.

In [ ]:
import socketserver

class EchoHandler(socketserver.BaseRequestHandler):
    def handle(self) -> None:
        data = self.request.recv(1024)
        self.request.sendall(data)

# server = socketserver.TCPServer(('127.0.0.1', 0), EchoHandler)
# server.handle_request()  # gere une seule requete
# server.server_close()
print('socketserver disponible')

---

## 10. `struct` -- encodage binaire

Pour les protocoles binaires, `struct` permet de packer/unpacker des donnees.

In [ ]:
import struct

# Packer un entier 32 bits et un float 64 bits
packed = struct.pack('>if', 42, 3.14)  # big-endian
packed

In [ ]:
struct.unpack('>if', packed)

`>` = big-endian, `i` = int32, `f` = float32, `d` = float64, `H` = uint16.

---

## Synthese

| Concept | A retenir |
|---|---|
| `socket.socket` | Base de toute communication reseau |
| TCP (`SOCK_STREAM`) | Fiable, ordonne, necessite `connect`/`accept` |
| UDP (`SOCK_DGRAM`) | Rapide, sans connexion, `sendto`/`recvfrom` |
| Prefixe de longueur | Protocole pour delimiter les messages sur TCP |
| `selectors` | Multiplexage I/O sans threads |
| `socketserver` | Classes de base pour serveurs simples |
| `struct` | Encodage/decodage binaire |

### Regles a retenir

- **TCP** pour la fiabilite, **UDP** pour la vitesse.
- **Toujours fermer les sockets** (`with` ou `close`).
- **Definir un protocole** de delimitation des messages sur TCP.
- **`selectors`** pour le multiplexage, pas des threads par client.
- **`struct`** pour les protocoles binaires.

---

## Exercices

### Exercice 1 -- Chat TCP simple

Ecrivez un serveur TCP qui recoit un message et renvoie le message en majuscules. Testez avec un client.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import socket, threading

def upper_server(port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.bind(('127.0.0.1', port))
    srv.listen(1)
    conn, _ = srv.accept()
    with conn:
        data = conn.recv(1024)
        conn.sendall(data.upper())
    srv.close()

PORT = 9999
threading.Thread(target=upper_server, args=(PORT,), daemon=True).start()

import time; time.sleep(0.1)
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(('127.0.0.1', PORT))
    s.sendall(b'hello world')
    print(s.recv(1024))
```

</details>

### Exercice 2 -- Client DNS simplifie

Utilisez `socket.getaddrinfo` pour resoudre un nom de domaine et afficher toutes les adresses IP associees.

In [ ]:
def resolve_dns(hostname: str) -> list[str]:
    ...

<details>
<summary>Voir la correction</summary>

```python
import socket

def resolve_dns(hostname: str) -> list[str]:
    infos = socket.getaddrinfo(hostname, None)
    ips = sorted(set(info[4][0] for info in infos))
    return ips

resolve_dns('google.com')
```

</details>

---

## Ressources externes

- [Documentation socket](https://docs.python.org/3/library/socket.html)
- [Socket Programming HOWTO](https://docs.python.org/3/howto/sockets.html)
- [Documentation socketserver](https://docs.python.org/3/library/socketserver.html)
- [Documentation selectors](https://docs.python.org/3/library/selectors.html)